# Chain-of-Thought Generator — Stage 0 (Zero Demonstrations)

Generates a `cot` column (natural-language derive-style reasoning) for every row
in `data/result_examples_0_result_k_6_balanced.parquet` and writes the result to
`data/result_examples_0_result_k_6_balanced_cot.parquet`.

Template specification: `docs/cot-template.md`

## 1. Setup & Imports

In [ ]:
import json
import re
from collections import Counter

import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

from src.encoding import encode_answer, _int_to_base_digits, _is_reversed_digit_mode
from src.operations import OPERATIONS
from src.op_index import SIGNED_OPS, CONCAT_OPS


# ---------------------------------------------------------------------------
# Equation formatting helpers (mirrors new_raw_dataset.ipynb)
# ---------------------------------------------------------------------------

def lhs_str(lhs: dict) -> str:
    return lhs["l0"] + lhs["l1"] + lhs["op"] + lhs["r0"] + lhs["r1"]


def rhs_str(rh: dict) -> str:
    body = "".join(rh["actual_rhs"])
    if rh["op"] is None:
        return body
    return rh["op"] + body if rh["sign_pos"] == "prefix" else body + rh["op"]


# ---------------------------------------------------------------------------
# JSON-decode guard (handles both string and already-decoded values)
# ---------------------------------------------------------------------------

def safe_json(v, default=None):
    if isinstance(v, str):
        return json.loads(v)
    if isinstance(v, float) and pd.isna(v):
        return default
    return v


print("Imports OK")

## 2. Load Dataset

In [ ]:
PARQUET_IN  = "data/result_examples_0_result_k_6_balanced.parquet"
PARQUET_OUT = "data/result_examples_0_result_k_6_balanced_cot.parquet"

pf = pq.ParquetFile(PARQUET_IN)
print("Schema:", pf.schema_arrow)

df = pq.read_table(PARQUET_IN).to_pandas()

# Decode JSON-string columns
for col in ["equations", "ops_mapping", "digits_mapping", "ops_candidates"]:
    df[col] = df[col].apply(lambda v: safe_json(v))

# Use solver_category from the new schema (fall back to deriving from solver_strategy for older parquets).
STRATEGY_TO_CATEGORY = {
    "pure_concat":   "concatenation",
    "standard":      "arithmetic",
    "little_endian": "arithmetic",
}
df["solver_category"] = df["solver_strategy"].map(STRATEGY_TO_CATEGORY)
assert df["solver_category"].notna().all(), (
    "Unmapped solver_strategy values: "
    + str(df.loc[df["solver_category"].isna(), "solver_strategy"].unique())
)

print(f"Loaded {len(df):,} rows. Columns: {list(df.columns)}")
print(df["solver_strategy"].value_counts())
print(df["solver_category"].value_counts())

assert df["solver_correct"].all(), "Unexpected wrong-answer rows!"
assert len(df) > 0, "Empty dataset!"

## 3. Static CoT Blocks

These blocks are identical for every row. Built once from `src/operations.py` so they stay
in lockstep if operations ever change.

In [ ]:
# ---------------------------------------------------------------------------
# Per-operation metadata (formula and guard in natural language)
# NOTE: maintain in lockstep with src/operations.py OPERATIONS dict ordering.
# ---------------------------------------------------------------------------

OP_FORMULA_NL = {
    "mul":        "a × b",
    "add":        "a + b",
    "absdiff":    "|a − b|",
    "sub_signed": "a − b  (result may be negative; if so, the operator symbol is attached to the answer as a sign)",
    "rsub_signed":"b − a  (same sign convention as sub_signed)",
    "neg_absdiff":"−|a − b|  (always non-positive; the operator symbol is always attached to the answer)",
    "concat_fwd": "the symbols of a followed by the symbols of b  (structural, no arithmetic)",
    "concat_rev": "the symbols of b followed by the symbols of a  (structural, no arithmetic)",
    "add_m1":     "a + b − 1  (invalid if result would be negative)",
    "add_p1":     "a + b + 1",
    "add_p2":     "a + b + 2",
    "mul_m1":     "a × b − 1  (invalid if result would be negative)",
    "mul_p1":     "a × b + 1",
    "absdiff_m1": "|a − b| − 1  (invalid when a = b)",
    "absdiff_p1": "|a − b| + 1  (always at least 1)",
    "absdiff_m2": "|a − b| − 2  (invalid when |a − b| < 2)",
    "gcd":        "greatest common divisor of a and b  (always non-negative)",
    "lcm":        "least common multiple of a and b  (0 if either operand is 0)",
    "mod":        "a mod b  (invalid when b = 0)",
    "rmod":       "b mod a  (invalid when a = 0)",
    "fdiv":       "⌊a ÷ b⌋  (invalid when b = 0)",
    "a2_plus_b":  "a² + b",
}

# Short formula for inline numeric expressions
OP_FORMULA_SHORT = {
    "mul":        "a × b",
    "add":        "a + b",
    "absdiff":    "|a − b|",
    "sub_signed": "a − b (signed)",
    "rsub_signed":"b − a (signed)",
    "neg_absdiff":"−|a − b|",
    "concat_fwd": "symbols of a + symbols of b",
    "concat_rev": "symbols of b + symbols of a",
    "add_m1":     "a + b − 1",
    "add_p1":     "a + b + 1",
    "add_p2":     "a + b + 2",
    "mul_m1":     "a × b − 1",
    "mul_p1":     "a × b + 1",
    "absdiff_m1": "|a − b| − 1",
    "absdiff_p1": "|a − b| + 1",
    "absdiff_m2": "|a − b| − 2",
    "gcd":        "gcd(a, b)",
    "lcm":        "lcm(a, b)",
    "mod":        "a mod b",
    "rmod":       "b mod a",
    "fdiv":       "⌊a ÷ b⌋",
    "a2_plus_b":  "a² + b",
}

assert set(OP_FORMULA_NL.keys()) == set(OPERATIONS.keys()), (
    "OP_FORMULA_NL is out of sync with OPERATIONS: "
    + str(set(OPERATIONS.keys()) ^ set(OP_FORMULA_NL.keys()))
)


# ---------------------------------------------------------------------------
# Build OPS_BLOCK — numbered list of all 22 ops
# ---------------------------------------------------------------------------

def make_ops_block() -> str:
    lines = ["Let's assume the following set of available operations that could govern the secret equations:\n"]
    for i, name in enumerate(OPERATIONS.keys(), 1):   # insertion order = OPERATIONS order
        formula = OP_FORMULA_NL[name]
        lines.append(f"{i:2d}. {name}(a, b) = {formula}")
    return "\n".join(lines)


OPS_BLOCK = make_ops_block()
print(OPS_BLOCK)

In [ ]:
# ---------------------------------------------------------------------------
# STRATEGIES_BLOCK — strategies + base + LHS/RHS structure + objective
# Fixed text; identical for every row.
# ---------------------------------------------------------------------------

STRATEGIES_BLOCK = """\
There are three possible solving strategies:

- Pure concatenation: the operator simply joins the two operands. The result is either the
  symbols of the left operand followed by those of the right operand (forward), or the symbols
  of the right operand followed by those of the left (reverse). No digit mapping is needed.

- Standard arithmetic (base 10, most-significant digit first): each unique symbol maps to a
  unique decimal digit (0-9). The left operand is decoded as L = digit(l0) x 10 + digit(l1),
  and the right operand as R = digit(r0) x 10 + digit(r1). The chosen operation is applied:
  result = OP(L, R). The magnitude is then converted to decimal digits from most to least
  significant, zero-padded on the left if the answer needs more digits, and each digit is
  mapped back to its symbol.

- Little-endian arithmetic (base 10, least-significant digit first): the same as standard
  arithmetic, except each operand's two digits are swapped before the operation -- the symbol
  in the first position carries the units digit (not the tens digit), so the raw two-digit
  value X is interpreted as (X mod 10) x 10 + (X / 10). The result digit sequence is
  reversed after zero-padding before mapping back to symbols.

We always use base 10.

The left-hand side of any equation has exactly 5 symbols: l0 l1 op r0 r1, where op is the
operator symbol, l0 l1 is the left operand, and r0 r1 is the right operand.

The right-hand side can have 1 to 4 symbols:
  - mag       -- all symbols represent the magnitude of the result.
  - op mag    -- operator symbol first, then magnitude (result is negative; sign prefix).
  - mag op    -- magnitude first, operator symbol last (result is negative; sign postfix).

The goal is to find the digit mapping (symbol -> digit) and operation mapping (op -> function)
that satisfies the equation, then apply them to decode the query answer."""

print(STRATEGIES_BLOCK)

## 4. Per-Row CoT Helpers

In [ ]:
# ---------------------------------------------------------------------------
# Symbol frequency ordering
# Returns content symbols (non-op) ordered by descending occurrence count;
# ties broken by first appearance (stable sort).
# ---------------------------------------------------------------------------

def symbol_freq_order(eq: dict) -> list[str]:
    lhs = eq["lhs"]
    op_char = lhs["op"]
    all_chars = (
        [lhs["l0"], lhs["l1"], lhs["r0"], lhs["r1"]]
        + [c for c in eq["rhs"]["actual_rhs"] if c != op_char]
    )
    # First-appearance order
    seen: dict[str, None] = {}
    for c in all_chars:
        if c not in seen:
            seen[c] = None
    ordered = list(seen.keys())
    cnt = Counter(all_chars)
    # Stable sort by descending count keeps first-appearance order for ties
    return sorted(ordered, key=lambda s: -cnt[s])


# ---------------------------------------------------------------------------
# Numeric helpers
# ---------------------------------------------------------------------------

def decode_lhs(eq: dict, dm: dict, radix: int = 10) -> tuple[int, int]:
    lhs = eq["lhs"]
    L = dm[lhs["l0"]] * radix + dm[lhs["l1"]]
    R = dm[lhs["r0"]] * radix + dm[lhs["r1"]]
    return L, R


def swap_digits(x: int, radix: int = 10) -> int:
    return (x % radix) * radix + x // radix


def format_op_call(op_name: str, L: int, R: int) -> str:
    """Human-readable arithmetic expression, e.g. '63 x 92', '|63 - 92|'."""
    d = {
        "mul":        f"{L} x {R}",
        "add":        f"{L} + {R}",
        "absdiff":    f"|{L} - {R}|",
        "sub_signed": f"{L} - {R}",
        "rsub_signed":f"{R} - {L}",
        "neg_absdiff":f"-|{L} - {R}|",
        "add_m1":     f"{L} + {R} - 1",
        "add_p1":     f"{L} + {R} + 1",
        "add_p2":     f"{L} + {R} + 2",
        "mul_m1":     f"{L} x {R} - 1",
        "mul_p1":     f"{L} x {R} + 1",
        "absdiff_m1": f"|{L} - {R}| - 1",
        "absdiff_p1": f"|{L} - {R}| + 1",
        "absdiff_m2": f"|{L} - {R}| - 2",
        "gcd":        f"gcd({L}, {R})",
        "lcm":        f"lcm({L}, {R})",
        "mod":        f"{L} mod {R}",
        "rmod":       f"{R} mod {L}",
        "fdiv":       f"floor({L} / {R})",
        "a2_plus_b":  f"{L}^2 + {R}",
    }
    return d.get(op_name, f"{op_name}({L}, {R})")


print("Helpers defined")

In [ ]:
# ---------------------------------------------------------------------------
# FAKE STEPS
# ---------------------------------------------------------------------------

def build_fake_steps_standard_unsigned(eq: dict, dm: dict, om: dict) -> str:
    """Two wrong-then-pruned steps for standard-arithmetic unsigned rows."""
    lhs = eq["lhs"]
    actual_rhs = eq["rhs"]["actual_rhs"]
    ans_str = "".join(actual_rhs)
    fwd = lhs["l0"] + lhs["l1"] + lhs["r0"] + lhs["r1"]
    rev = lhs["r0"] + lhs["r1"] + lhs["l0"] + lhs["l1"]

    step1 = (
        f"First, I'll check whether this could be a concatenation rule. "
        f"Forward concatenation (l0 l1 r0 r1) would give {fwd}. "
        f"Reverse concatenation (r0 r1 l0 l1) would give {rev}. "
        f"Neither matches the answer {ans_str}, so concatenation is ruled out."
    )
    step2 = (
        f"Next, could the operation be a negated difference? That operation always produces a "
        f"non-positive result and always attaches the operator symbol to the front of the answer. "
        f"The answer {ans_str} carries no such prefix, so a negated difference is impossible here."
    )
    return step1 + "\n\n" + step2


def build_fake_steps_little_endian_unsigned(eq: dict, dm: dict, om: dict) -> str:
    """Two wrong-then-pruned steps for little_endian unsigned rows.

    Step 1: rule out concat.
    Step 2: show standard arithmetic fails with the true (LE) digit mapping.
    """
    lhs = eq["lhs"]
    op_char = lhs["op"]
    op_name = om[op_char]
    actual_rhs = eq["rhs"]["actual_rhs"]
    ans_str = "".join(actual_rhs)
    fwd = lhs["l0"] + lhs["l1"] + lhs["r0"] + lhs["r1"]
    rev = lhs["r0"] + lhs["r1"] + lhs["l0"] + lhs["l1"]

    step1 = (
        f"First, checking concatenation. "
        f"Forward gives {fwd} and reverse gives {rev}. "
        f"Neither matches {ans_str}. Concatenation is ruled out."
    )

    # Try standard arithmetic with the truth LE mapping — always fails for LE rows
    freq = symbol_freq_order(eq)
    assign_str = ", ".join(f"{s} -> {dm[s]}" for s in freq if s in dm)
    L, R = decode_lhs(eq, dm)
    res_std = OPERATIONS.get(op_name, lambda a, b: None)(L, R)
    res_len = len(actual_rhs)

    if res_std is not None:
        digits_std = _int_to_base_digits(abs(res_std) if op_name in SIGNED_OPS else res_std, 10)
        padded_std = [0] * (res_len - len(digits_std)) + digits_std
        inv = {v: k for k, v in dm.items()}
        decoded_std = "".join(inv.get(d, f"[no symbol for {d}]") for d in padded_std)

        step2 = (
            f"I'll assign digits starting from the most frequent symbols: {assign_str}.\n"
            f"Left operand raw value: {dm[lhs['l0']]} x 10 + {dm[lhs['l1']]} = {L}.\n"
            f"Right operand raw value: {dm[lhs['r0']]} x 10 + {dm[lhs['r1']]} = {R}.\n"
            f"Trying standard arithmetic ({OP_FORMULA_SHORT[op_name]}): "
            f"{format_op_call(op_name, L, R)} = {res_std}. "
            f"Digits {padded_std} would map to {decoded_std}. "
            f"This does not match the expected answer {ans_str}, so standard "
            f"encoding fails here."
        )
    else:
        step2 = (
            f"Trying standard arithmetic gives an invalid or undefined result for this "
            f"assignment. Standard encoding does not apply here."
        )

    return step1 + "\n\n" + step2


def build_fake_steps_signed(eq: dict, dm: dict, om: dict, strategy: str) -> str:
    """Two wrong-then-pruned steps for signed rows (prefix or postfix)."""
    lhs = eq["lhs"]
    op_char = lhs["op"]
    op_name = om[op_char]
    sign_pos = eq["rhs"]["sign_pos"]
    actual_rhs = eq["rhs"]["actual_rhs"]
    mag_str = "".join(actual_rhs)
    full_ans = (op_char + mag_str) if sign_pos == "prefix" else (mag_str + op_char)
    sign_desc = "at the start" if sign_pos == "prefix" else "at the end"

    step1 = (
        f"The answer {full_ans} has the operator symbol {op_char} {sign_desc} — "
        f"concatenation always produces bare symbol sequences with no sign character. "
        f"Concatenation is ruled out."
    )

    # Compute absdiff for comparison (always non-negative, no sign)
    L, R = decode_lhs(eq, dm)
    if strategy == "little_endian":
        Lc, Rc = swap_digits(L), swap_digits(R)
    else:
        Lc, Rc = L, R

    abs_result = abs(Lc - Rc)
    abs_digits = _int_to_base_digits(abs_result, 10)
    res_len = len(actual_rhs)
    abs_padded = [0] * (res_len - len(abs_digits)) + abs_digits
    inv = {v: k for k, v in dm.items()}
    abs_decoded = "".join(inv.get(d, f"[?{d}]") for d in abs_padded)

    step2 = (
        f"Could this be an absolute difference? |{Lc} - {Rc}| = {abs_result} "
        f"-> symbols {abs_decoded}. "
        f"But the answer {full_ans} carries a sign character — "
        f"an absolute difference is always non-negative and never adds a sign. "
        f"The sign signals a signed subtraction where the result went below zero."
    )

    return step1 + "\n\n" + step2


def build_fake_steps(eq: dict, dm: dict, om: dict, strategy: str) -> str:
    """Dispatch fake steps based on sign_pos and strategy."""
    sign_pos = eq["rhs"]["sign_pos"]
    if sign_pos is not None:
        return build_fake_steps_signed(eq, dm, om, strategy)
    elif strategy == "little_endian":
        return build_fake_steps_little_endian_unsigned(eq, dm, om)
    else:
        return build_fake_steps_standard_unsigned(eq, dm, om)


print("Fake-step builders defined")

In [ ]:
# ---------------------------------------------------------------------------
# TRUTH REASONING
# ---------------------------------------------------------------------------

def build_truth_standard(eq: dict, dm: dict, om: dict, answer: str) -> str:
    """Derive-style truth reasoning for standard arithmetic."""
    lhs = eq["lhs"]
    op_char = lhs["op"]
    op_name = om[op_char]
    actual_rhs = eq["rhs"]["actual_rhs"]
    sign_pos = eq["rhs"]["sign_pos"]
    res_len = len(actual_rhs)
    inv = {v: k for k, v in dm.items()}

    freq = symbol_freq_order(eq)
    cnt = Counter(
        [lhs["l0"], lhs["l1"], lhs["r0"], lhs["r1"]]
        + [c for c in actual_rhs if c != op_char]
    )

    L, R = decode_lhs(eq, dm)
    result_raw = OPERATIONS[op_name](L, R)

    if op_name in SIGNED_OPS or op_name == "neg_absdiff":
        mag = abs(result_raw)
    else:
        mag = result_raw

    digits = _int_to_base_digits(mag, 10)
    needs_pad = len(digits) < res_len
    padded = [0] * (res_len - len(digits)) + digits
    max_val = 10 ** res_len

    parts = []

    parts.append("**Finding the digit and operation mapping using standard arithmetic (base 10).**")

    # Symbol frequency
    freq_lines = [f"  {s}: {cnt[s]} occurrence{'s' if cnt[s] > 1 else ''}" for s in freq]
    parts.append(
        "I'll prioritise assigning digits to the symbols that appear most often — "
        "the more frequently a symbol appears, the more it constrains the solution:\n"
        + "\n".join(freq_lines)
    )

    # Digit assignment
    assign_lines = [f"  {s} -> {dm[s]}" for s in freq]
    parts.append("Assigning digits:\n" + "\n".join(assign_lines))

    # Operand decode
    parts.append(
        f"Left operand:  digit({lhs['l0']}) x 10 + digit({lhs['l1']}) = "
        f"{dm[lhs['l0']]} x 10 + {dm[lhs['l1']]} = **{L}**.\n"
        f"Right operand: digit({lhs['r0']}) x 10 + digit({lhs['r1']}) = "
        f"{dm[lhs['r0']]} x 10 + {dm[lhs['r1']]} = **{R}**."
    )

    # Apply op + overflow guard
    parts.append(
        f"Applying the operation ({OP_FORMULA_SHORT[op_name]}): "
        f"{format_op_call(op_name, L, R)} = **{result_raw}**.\n"
        f"The result magnitude {mag} fits within {res_len} position(s) ({mag} < {max_val}). ✓"
    )

    # Zero-padding
    if needs_pad:
        parts.append(
            f"The magnitude {mag} has {len(digits)} digit(s) but the answer has {res_len} "
            f"symbol position(s). Padding the leading position(s) with the symbol for zero gives "
            f"digits {padded}."
        )

    # Map back
    sym_map_parts = [f"{d} -> {inv[d]}" for d in padded]
    mag_symbols = "".join(inv[d] for d in padded)
    parts.append(
        "Converting digits back to symbols: "
        + ", ".join(sym_map_parts)
        + f" -> **{mag_symbols}**."
    )

    # Sign
    if sign_pos == "prefix":
        parts.append(
            f"The result is {'negative' if result_raw < 0 else 'non-positive'}, so the operator "
            f"symbol {op_char} is prepended to give the final answer."
        )
    elif sign_pos == "postfix":
        parts.append(
            f"The result is {'negative' if result_raw < 0 else 'non-positive'}, so the operator "
            f"symbol {op_char} is appended to give the final answer."
        )

    # Summary
    dm_str = ", ".join(f"{s} -> {d}" for s, d in dm.items())
    parts.append(
        f"**Summary:**\n"
        f"- Digit mapping: {dm_str}\n"
        f"- Operation: {op_char} -> {op_name} ({OP_FORMULA_SHORT[op_name]})"
    )

    # Generic underdetermination note
    parts.append(
        "Note: this equation may admit more than one consistent mapping; "
        "the solution shown above is one valid program that satisfies it."
    )


    return "\n\n".join(parts)


def build_truth_little_endian(eq: dict, dm: dict, om: dict, answer: str) -> str:
    """Derive-style truth reasoning for little-endian arithmetic."""
    lhs = eq["lhs"]
    op_char = lhs["op"]
    op_name = om[op_char]
    actual_rhs = eq["rhs"]["actual_rhs"]
    sign_pos = eq["rhs"]["sign_pos"]
    res_len = len(actual_rhs)
    inv = {v: k for k, v in dm.items()}

    freq = symbol_freq_order(eq)
    cnt = Counter(
        [lhs["l0"], lhs["l1"], lhs["r0"], lhs["r1"]]
        + [c for c in actual_rhs if c != op_char]
    )

    L, R = decode_lhs(eq, dm)
    Ls, Rs = swap_digits(L), swap_digits(R)
    result_raw = OPERATIONS[op_name](Ls, Rs)

    if op_name in SIGNED_OPS or op_name == "neg_absdiff":
        mag = abs(result_raw)
    else:
        mag = result_raw

    digits = _int_to_base_digits(mag, 10)
    needs_pad = len(digits) < res_len
    padded = [0] * (res_len - len(digits)) + digits
    reversed_digits = padded[::-1]
    max_val = 10 ** res_len

    parts = []

    parts.append("**Finding the digit and operation mapping using little-endian arithmetic (base 10).**")

    parts.append(
        "In little-endian encoding, each two-symbol operand is read with its digit pair reversed: "
        "the first symbol carries the units digit (not the tens digit), and the second carries the tens. "
        "If the raw two-digit value is X, the interpreted value is (X mod 10) x 10 + (X / 10). "
        "The result digit sequence is also reversed after zero-padding before mapping back to symbols."
    )

    # Symbol frequency
    freq_lines = [f"  {s}: {cnt[s]}" for s in freq]
    parts.append("Symbol frequencies:\n" + "\n".join(freq_lines))

    # Digit assignment
    assign_lines = [f"  {s} -> {dm[s]}" for s in freq]
    parts.append("Assigning digits:\n" + "\n".join(assign_lines))

    # Operand decode + swap
    parts.append(
        f"Raw two-digit values from the LHS:\n"
        f"  Left:  digit({lhs['l0']}) x 10 + digit({lhs['l1']}) = "
        f"{dm[lhs['l0']]} x 10 + {dm[lhs['l1']]} = {L}\n"
        f"  Right: digit({lhs['r0']}) x 10 + digit({lhs['r1']}) = "
        f"{dm[lhs['r0']]} x 10 + {dm[lhs['r1']]} = {R}\n\n"
        f"Applying the digit swap (units <-> tens):\n"
        f"  Left:  {L} -> ({L} mod 10) x 10 + ({L} / 10) = {L % 10} x 10 + {L // 10} = **{Ls}**\n"
        f"  Right: {R} -> ({R} mod 10) x 10 + ({R} / 10) = {R % 10} x 10 + {R // 10} = **{Rs}**"
    )

    # Apply op + overflow guard
    parts.append(
        f"Applying the operation ({OP_FORMULA_SHORT[op_name]}): "
        f"{format_op_call(op_name, Ls, Rs)} = **{result_raw}**.\n"
        f"The result magnitude {mag} fits within {res_len} position(s) ({mag} < {max_val}). ✓"
    )

    # Pad + reverse
    if needs_pad:
        parts.append(
            f"The magnitude {mag} has {len(digits)} digit(s) but the answer has {res_len} "
            f"position(s). Padding gives {padded}. Reversing for little-endian: {reversed_digits}."
        )
    else:
        parts.append(
            f"Digits {padded} -> reversed for little-endian: {reversed_digits}."
        )

    # Map back
    sym_map_parts = [f"{d} -> {inv[d]}" for d in reversed_digits]
    mag_symbols = "".join(inv[d] for d in reversed_digits)
    parts.append(
        "Mapping digits back to symbols: "
        + ", ".join(sym_map_parts)
        + f" -> **{mag_symbols}**."
    )

    # Sign
    if sign_pos == "prefix":
        parts.append(
            f"The result is {'negative' if result_raw < 0 else 'non-positive'}, so the operator "
            f"symbol {op_char} is prepended to give the final answer."
        )
    elif sign_pos == "postfix":
        parts.append(
            f"The result is {'negative' if result_raw < 0 else 'non-positive'}, so the operator "
            f"symbol {op_char} is appended to give the final answer."
        )

    # Summary
    dm_str = ", ".join(f"{s} -> {d}" for s, d in dm.items())
    parts.append(
        f"**Summary:**\n"
        f"- Digit mapping: {dm_str}\n"
        f"- Operation: {op_char} -> {op_name} ({OP_FORMULA_SHORT[op_name]})\n"
        f"- Strategy: little-endian"
    )

    # Generic underdetermination note
    parts.append(
        "Note: this equation may admit more than one consistent mapping; "
        "the solution shown above is one valid program that satisfies it."
    )


    return "\n\n".join(parts)


print("Truth-reasoning builders defined")

In [ ]:
# ---------------------------------------------------------------------------
# CONCATENATION CoT
# ---------------------------------------------------------------------------

def build_concat_cot(eq: dict, answer: str) -> str:
    lhs = eq["lhs"]
    fwd = lhs["l0"] + lhs["l1"] + lhs["r0"] + lhs["r1"]
    rev = lhs["r0"] + lhs["r1"] + lhs["l0"] + lhs["l1"]

    parts = []
    parts.append(
        "Let me check whether the operator in this equation acts as a concatenation rule."
    )
    if answer == fwd:
        parts.append(
            f"Forward concatenation (left operand followed by right operand) gives {fwd}. "
            f"That matches the answer exactly. "
            f"No digit mapping is needed — the operator simply glues the operands left-to-right."
        )
    elif answer == rev:
        parts.append(
            f"Forward concatenation would give {fwd}. That does not match."
        )
        parts.append(
            f"Reverse concatenation (right operand followed by left operand) gives {rev}. "
            f"That matches the answer. "
            f"No digit mapping is needed — the operator glues the operands right-to-left."
        )
    else:
        # Fallback: shouldn't happen for category == 'concatenation'
        parts.append(
            f"Forward concatenation gives {fwd}; reverse gives {rev}. "
            f"The answer {answer} matches one of these concatenation forms."
        )

    return "\n\n".join(parts)


# ---------------------------------------------------------------------------
# OBSERVATIONS block
# ---------------------------------------------------------------------------

def build_observations(eq: dict, answer: str, category: str) -> str:
    lhs = eq["lhs"]
    rhs = eq["rhs"]
    op_char = lhs["op"]
    sign_pos = rhs["sign_pos"]
    actual_rhs = rhs["actual_rhs"]
    res_len = len(actual_rhs)

    bullets = []

    if sign_pos == "prefix":
        mag_str = "".join(actual_rhs)
        bullets.append(
            f"The answer {answer} starts with the operator symbol {op_char}, "
            f"indicating the result is negative (sign prefix). "
            f"The magnitude part has {res_len} symbol(s): {mag_str}."
        )
    elif sign_pos == "postfix":
        mag_str = "".join(actual_rhs)
        bullets.append(
            f"The answer {answer} ends with the operator symbol {op_char}, "
            f"indicating the result is negative (sign postfix). "
            f"The magnitude part has {res_len} symbol(s): {mag_str}."
        )
    else:
        bullets.append(
            f"The answer {answer} carries no operator symbol, so the result is "
            f"non-negative and no signed operation is in play."
        )

    if sign_pos is None:
        if res_len == 4:
            bullets.append(
                f"The answer has 4 symbols, matching the combined length of the two operands "
                f"(2 + 2). This is consistent with either a 4-digit arithmetic result or "
                f"concatenation."
            )
        else:
            bullets.append(
                f"The answer has {res_len} symbol(s) — fewer than 4, so concatenation "
                f"(which always produces 4 symbols) is already ruled out."
            )

    return "Observations:\n" + "\n".join(f"- {b}" for b in bullets)


print("Concat and observations builders defined")

In [ ]:
# ---------------------------------------------------------------------------
# TOP-LEVEL: build_cot(row)
# ---------------------------------------------------------------------------

def build_cot(row) -> str:
    eqs          = row["equations"]       # already decoded list
    eq           = eqs[0]                 # single to_predict equation
    category     = row["solver_category"]
    strategy     = row["solver_strategy"]
    dm           = row["digits_mapping"] if isinstance(row["digits_mapping"], dict) else {}
    om           = row["ops_mapping"]    if isinstance(row["ops_mapping"],    dict) else {}
    answer       = row["answer"]

    eq_line = f"  {lhs_str(eq['lhs'])} = {rhs_str(eq['rhs'])}"
    obs     = build_observations(eq, answer, category)

    sections = [
        OPS_BLOCK,
        STRATEGIES_BLOCK,
        f"Formatted equation:\n{eq_line}",
        obs,
    ]

    if category == "concatenation":
        sections.append(build_concat_cot(eq, answer))
        # Verify concat answer matches forward or reverse concatenation
        lhs = eq["lhs"]
        fwd = lhs["l0"] + lhs["l1"] + lhs["r0"] + lhs["r1"]
        rev = lhs["r0"] + lhs["r1"] + lhs["l0"] + lhs["l1"]
        assert answer in (fwd, rev), (
            f"{row['problem_id']}: concat answer {answer!r} matches neither fwd {fwd!r} nor rev {rev!r}"
        )
        # ops_mapping label consistency: glyph must match lhs op; label must match answer direction
        if om:
            glyph, label = next(iter(om.items()))
            assert glyph == eq["lhs"]["op"], (
                f"{row['problem_id']}: ops_mapping glyph {glyph!r} != lhs op {eq['lhs']['op']!r}"
            )
            assert label in ("concat_fwd", "concat_rev"), (
                f"{row['problem_id']}: unexpected concat label {label!r}"
            )
            if fwd != rev:  # ambiguous rows (fwd == rev) cannot be attributed
                expected = "concat_fwd" if answer == fwd else "concat_rev"
                assert label == expected, (
                    f"{row['problem_id']}: ops_mapping says {label!r} but answer matches {expected!r}"
                )
    else:
        # Fake steps
        sections.append(build_fake_steps(eq, dm, om, strategy))
        # Truth reasoning
        if strategy == "little_endian":
            sections.append(build_truth_little_endian(eq, dm, om, answer))
        else:
            sections.append(build_truth_standard(eq, dm, om, answer))
        # Verify the derived answer matches the gold answer
        op_char = eq["lhs"]["op"]
        mode = "little_endian" if strategy == "little_endian" else "standard"
        enc = encode_answer(dm, om, op_char, eq["lhs"], mode, 10, answer)
        assert enc is not None and enc[0] == answer, (
            f"{row['problem_id']}: reconstructed answer {enc!r} != gold {answer!r}"
        )

    sections.append(f"solution: {answer}")

    return "\n\n".join(sections)


print("build_cot defined")

## 5. Spot-Check

Render one example of each strategy variant and inspect the generated CoT.

In [ ]:
def spot_check(label: str, row):
    cot = build_cot(row)
    print(f"\n{'='*70}")
    print(f"[{label}] id={row['problem_id']}  strategy={row['solver_strategy']}  "
          f"category={row['solver_category']}  answer={repr(row['answer'])}")
    print('='*70)
    print(cot)


# Standard unsigned
row_std = df[(df["solver_strategy"] == "standard") &
             (df["equations"].apply(lambda e: e[0]["rhs"]["sign_pos"] is None))].iloc[0]
spot_check("STANDARD UNSIGNED", row_std)


In [ ]:
# Little-endian unsigned
row_le = df[(df["solver_strategy"] == "little_endian") &
            (df["equations"].apply(lambda e: e[0]["rhs"]["sign_pos"] is None))].iloc[0]
spot_check("LITTLE-ENDIAN UNSIGNED", row_le)

In [ ]:
# Signed prefix (standard)
row_sp = df[df["equations"].apply(lambda e: e[0]["rhs"]["sign_pos"] == "prefix")].iloc[0]
spot_check("SIGNED PREFIX", row_sp)

In [ ]:
# Signed postfix
row_spost = df[df["equations"].apply(lambda e: e[0]["rhs"]["sign_pos"] == "postfix")].iloc[0]
spot_check("SIGNED POSTFIX", row_spost)

In [ ]:
# Little-endian signed
row_le_s = df[(df["solver_strategy"] == "little_endian") &
              (df["equations"].apply(lambda e: e[0]["rhs"]["sign_pos"] is not None))].iloc[0]
spot_check("LITTLE-ENDIAN SIGNED", row_le_s)

In [ ]:
# Pure concat (forward)
def _is_fwd_concat(row):
    eq = row["equations"][0]
    lhs = eq["lhs"]
    return row["answer"] == lhs["l0"] + lhs["l1"] + lhs["r0"] + lhs["r1"]

row_cc_fwd = df[(df["solver_category"] == "concatenation") & df.apply(_is_fwd_concat, axis=1)].iloc[0]
spot_check("PURE CONCAT FWD", row_cc_fwd)

In [ ]:
# Pure concat (reverse)
def _is_rev_concat(row):
    eq = row["equations"][0]
    lhs = eq["lhs"]
    return row["answer"] == lhs["r0"] + lhs["r1"] + lhs["l0"] + lhs["l1"]

rev_concat_rows = df[(df["solver_category"] == "concatenation") & df.apply(_is_rev_concat, axis=1)]
print(f"Reverse-concat rows: {len(rev_concat_rows)}")
if len(rev_concat_rows):
    spot_check("PURE CONCAT REV", rev_concat_rows.iloc[0])
else:
    print("No reverse-concat rows in this dataset.")

## 6. Generate All Rows

In [ ]:
print(f"Generating CoT for {len(df):,} rows...")
df["cot"] = df.apply(build_cot, axis=1)
print("Done.")

# Sanity checks
assert df["cot"].notna().all(), "Some CoT values are null!"
assert (df["cot"] != "").all(), "Some CoT values are empty!"

# Quick stats
lengths = df["cot"].str.len()
print(f"CoT length — min: {lengths.min()}, median: {lengths.median():.0f}, max: {lengths.max()}")

# Approximate token count (2.81 chars / token)
CHARS_PER_TOKEN = 2.81
df["cot_approx_tokens"] = (df["cot"].str.len() / CHARS_PER_TOKEN).round().astype(int)

tok = df["cot_approx_tokens"]
print(f"CoT approx tokens — min: {tok.min()}, median: {tok.median():.0f}, max: {tok.max()}")

# --- Voice-rule guard: no Python/Rust identifiers or heuristic labels in any CoT ---
FORBIDDEN_TOKENS = [
    "Heuristic", ".py", ".rs", "src/", "lib.rs", "def ",
    "encode_answer", "build_cot", "build_truth", "build_fake",
    "symbol_freq", "decode_lhs", "swap_digits", "OPERATIONS",
]
hits = df["cot"].apply(lambda c: [t for t in FORBIDDEN_TOKENS if t in c])
bad_voice = df[hits.apply(len) > 0]
assert bad_voice.empty, (
    f"Forbidden tokens found in CoT for rows: {bad_voice['id'].head().tolist()}\n"
    f"Tokens: {hits[hits.apply(len) > 0].head().tolist()}"
)
print("✓ Voice-rule check passed — no forbidden implementation tokens in any CoT")

# --- Strategy coverage: all three strategies must be represented ─────────────
present_strategies = set(df["solver_strategy"].unique())
assert {"standard", "little_endian", "pure_concat"} <= present_strategies, (
    f"Missing strategies in dataset: {present_strategies}"
)
print(f"✓ Strategy coverage OK: {sorted(present_strategies)}")

# --- Structural length checks ─────────────────────────────────────────────────
assert lengths.min() > 200, f"Suspiciously short CoT (min={lengths.min()})"
concat_median = lengths[df["solver_category"] == "concatenation"].median()
arith_median  = lengths[df["solver_category"] == "arithmetic"].median()
assert concat_median < arith_median, (
    f"Expected concat CoTs shorter than arithmetic (no fake-steps), "
    f"but concat median {concat_median:.0f} >= arithmetic median {arith_median:.0f}"
)
print(f"✓ Length sanity OK — concat median {concat_median:.0f} < arithmetic median {arith_median:.0f}")

## 7. Write Parquet

In [ ]:
# --- Pre-write guard: ensure generation ran and produced a full cot column ----
assert "cot" in df.columns, (
    "'cot' column missing — run Section 6 (Generate All Rows) before writing!"
)
assert df["cot"].notna().all() and (df["cot"] != "").all(), (
    "Null or empty CoT values present — regenerate before writing!"
)

table = pa.Table.from_pandas(df, preserve_index=False)
pq.write_table(table, PARQUET_OUT)
print(f"Written {len(df):,} rows to {PARQUET_OUT}")

# Round-trip check
df_check = pq.read_table(PARQUET_OUT).to_pandas()
assert len(df_check) == len(df), "Row count changed after round-trip!"
assert "cot" in df_check.columns, "'cot' column missing after round-trip!"
assert df_check["cot"].notna().all(), "Null CoT values after round-trip!"
print(f"Round-trip OK. Columns: {list(df_check.columns)}")
print("\nSample CoT (first 300 chars of row 0):")
print(df_check["cot"].iloc[0][:300])